# Horas punta de mantenimiento: sustento para reforzar el personal de almacen

**Objetivo:** identificar en que dias y horas los tecnicos empiezan mas actividades de mantenimiento, porque en esos momentos necesitan repuestos del almacen. Con eso se sustenta en que franjas se necesita **mas personal en almacen** para que los tecnicos no esperen.

**Fuente:** reporte *Bitacora de Mantenimiento* de Odoo (wizard `maintenance.bitacora.report.wizard`).

**Como usarlo:** ejecutar todas las celdas en orden (la descarga tarda ~5 minutos). Solo se cambian los valores de la seccion **1. Parametros**.

| Seccion | Que hace |
|---|---|
| 1. Parametros | Valores que se pueden cambiar |
| 2. Conexion a Odoo | Inicia sesion con el `.env` |
| 3. Descarga | Trae la bitacora mes por mes |
| 4. Limpieza | Deja la tabla lista (textos, numeros y horas de Lima) |
| 5. Preparacion | Se queda con las horas de inicio y arma los grupos |
| 6. Perfil por hora | Cuantas actividades empiezan en cada hora |
| 7. Mapa de calor | Lo mismo, separado por dia de la semana |
| 8. Franjas | Agrupa las horas en niveles Alta / Media / Baja |
| 9. Sustento | Pruebas de que las horas punta son reales y se repiten |
| 10. Conclusiones | Resumen en texto, listo para presentar |
| 11. Exportar | (Opcional) guarda los resultados en Excel |

## 0. Librerias

- `os` y `dotenv`: leen el usuario y la clave de Odoo desde el archivo `.env` (asi no quedan escritos en el codigo).
- `xmlrpc.client`: permite enviar pedidos a Odoo por XML-RPC.
- `pandas`: para trabajar con tablas (DataFrames).

In [ ]:
import os
from dotenv import load_dotenv
import xmlrpc.client
import pandas as pd

## 1. Parametros

Son los unicos valores que normalmente se cambian.

- **`EMPRESA_ID`**: `2` = TATSA, `1` = TVL.
- **`FECHA_INICIO_DESCARGA` / `FECHA_FIN_DESCARGA`**: rango que se pide a Odoo. El fin es *hoy*, asi siempre trae lo ultimo.
- **`INICIO_ANALISIS`**: aunque se descargue desde enero de 2025, el reporte recien se uso desde el **07/05/2025** (antes solo hay 15 filas de prueba en febrero). Se empieza el 08/05 para no contar un dia incompleto.
- **`SEGMENTO`**: que actividades se analizan:
  - `"Todas"`: todas.
  - `"Sin inspecciones"` *(recomendado)*: quita las actividades que empiezan con "INSPECCIÓN". La inspeccion diaria de neumaticos es ~26% de todo, se hace de noche y probablemente no pide repuestos; si se deja, hace parecer que la punta es de noche.
  - `"Correctivas"`: solo reparaciones.

In [ ]:
EMPRESA_ID = 2                                                  # 2 = TATSA, 1 = TVL
FECHA_INICIO_DESCARGA = "2025-01-01"
FECHA_FIN_DESCARGA = pd.Timestamp.today().strftime("%Y-%m-%d")  # hoy

INICIO_ANALISIS = "2025-05-08"                                  # primer dia completo con datos
SEGMENTO = "Sin inspecciones"                                   # "Todas", "Sin inspecciones" o "Correctivas"

## 2. Conexion a Odoo

1. `load_dotenv()` lee el archivo `.env` y `os.getenv(...)` toma cada valor.
2. `common.authenticate(...)` inicia sesion y devuelve tu numero de usuario (`uid`).
3. `models` es el objeto con el que se hacen todos los pedidos (`execute_kw`).

In [ ]:
load_dotenv()
url = os.getenv("URL_ODOO")
username = os.getenv("USER_JIM")
password = os.getenv("PASSWORD_JIM")
db = os.getenv("BD_ODOO")

common = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/common")
uid = common.authenticate(db, username, password, {})
print(f"uid conectado : {uid}")

models = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/object", allow_none=True)

## 3. Descarga de la bitacora

El reporte no esta guardado en una tabla: Odoo lo arma cuando presionas **Mostrar**. Por eso se imitan los 3 pasos de la pantalla:

| Paso | En pantalla | En Python |
|---|---|---|
| 1 | Llenar las fechas | `create` en el wizard → devuelve el id del wizard |
| 2 | Presionar **Mostrar** | `get_report` con ese id |
| 3 | Ver la lista | `search_read` en `maintenance.bitacora.report` |

Detalles:
- `[(6, 0, [])]` es un **comando** de Odoo para los campos con varios valores: *"usa exactamente esta lista"*. Lista vacia = sin filtro.
- En el paso 3 el contexto lleva `active_id`: le dice a Odoo de que wizard son los resultados (asi lo hace el navegador).
- No se pone `limit`, para traer **todas** las filas (el navegador pide solo 80 por pagina).
- Se descarga **mes por mes**: un mes son ~17.000 actividades (~15 segundos). Pedir todo junto (~300.000 filas) puede tardar mucho o cortarse.

In [ ]:
contexto = {"lang": "es_PE", "tz": "America/Lima", "allowed_company_ids": [EMPRESA_ID]}

campos = [
    "schedule_date", "vehicle_id", "vehicle_code", "request_id", "km_schedule", "km_final",
    "init_user_name", "final_user_name", "employee_name", "work_shift_id", "is_stranded",
    "reason_name", "template_name", "positions", "breakdown_date", "location",
    "breakdown_location", "component_system_name", "activity_name", "activity_duration",
    "activity_anotation", "maintenance_type", "activity_start_datetime",
    "activity_end_datetime", "implemented_hours", "vehicle_release_time",
]


def descargar_bitacora(fecha_inicio, fecha_fin):
    # Paso 1: crear el wizard con las fechas
    wizard_id = models.execute_kw(
        db, uid, password,
        "maintenance.bitacora.report.wizard", "create",
        [{
            "date_from": fecha_inicio,
            "date_to": fecha_fin,
            "filtrar_porfechainicio": False,
            "filtrar_porfechafin": False,
            "maintenance_type": "all",
            "content_vehicle": "all",
            "content_vehicle_type": "all",
            "content_request": "all",
            "content_activity_preventive": "all",
            "content_activity_corrective": "all",
            "vehicle_ids": [(6, 0, [])],
            "vehicle_type_ids": [(6, 0, [])],
            "request_ids": [(6, 0, [])],
            "activity_preventive_ids": [(6, 0, [])],
            "activity_corrective_ids": [(6, 0, [])],
        }],
        {"context": contexto}
    )

    # Paso 2: presionar "Mostrar"
    models.execute_kw(
        db, uid, password,
        "maintenance.bitacora.report.wizard", "get_report",
        [[wizard_id]],
        {"context": contexto}
    )

    # Paso 3: leer los resultados de ese wizard
    contexto_reporte = {
        **contexto,
        "active_id": wizard_id,
        "active_ids": [wizard_id],
        "active_model": "maintenance.bitacora.report.wizard",
    }
    return models.execute_kw(
        db, uid, password,
        "maintenance.bitacora.report", "search_read",
        [[]],
        {"fields": campos, "context": contexto_reporte}
    )


bitacora = []
for inicio_mes in pd.date_range(FECHA_INICIO_DESCARGA, FECHA_FIN_DESCARGA, freq="MS"):   # "MS" = primer dia de cada mes
    fin_mes = min(inicio_mes + pd.offsets.MonthEnd(0), pd.Timestamp(FECHA_FIN_DESCARGA))
    filas = descargar_bitacora(inicio_mes.strftime("%Y-%m-%d"), fin_mes.strftime("%Y-%m-%d"))
    bitacora += filas
    print(f"{inicio_mes:%Y-%m}: {len(filas):>6} actividades")

print("Total:", len(bitacora))

## 4. Limpieza

Primero se pregunta a Odoo el **tipo** de cada campo (`fields_get`) para limpiar cada columna segun corresponda:

- **Vacios:** Odoo manda `False` cuando un campo esta vacio. Se cambia por vacio (`None`), excepto en los campos si/no.
  Se usa `valor is False` y no `valor == False`, porque en Python **`0 == False` es verdadero**: con `==` se borrarian todos los ceros (por ejemplo, 0 horas ejecutadas).
- **many2one** (campos que apuntan a otra tabla): llegan como `[15, "Bus 101"]`; se deja solo el nombre.
- **Fechas con hora:** Odoo las entrega **en UTC** (5 horas adelante de Lima). Se convierten a hora de Lima; si no, todas las horas punta saldrian corridas 5 horas.

In [ ]:
info = models.execute_kw(
    db, uid, password,
    "maintenance.bitacora.report", "fields_get",
    [campos],
    {"attributes": ["string", "type"], "context": contexto}
)


def limpiar(valor, tipo):
    if valor is False and tipo != "boolean":
        return None
    if tipo == "many2one" and isinstance(valor, list):
        return valor[1]
    return valor


df_bitacora = pd.DataFrame(bitacora).drop(columns="id")

for campo in campos:
    tipo = info[campo]["type"]
    df_bitacora[campo] = df_bitacora[campo].apply(lambda v: limpiar(v, tipo))

    if tipo == "datetime":
        df_bitacora[campo] = (
            pd.to_datetime(df_bitacora[campo], utc=True)
            .dt.tz_convert("America/Lima")
            .dt.tz_localize(None)
        )

# Copia con los nombres que se ven en la pantalla de Odoo (para revisar o exportar)
df_bitacora_es = df_bitacora.rename(columns={c: info[c]["string"] for c in campos})
print("Filas:", len(df_bitacora_es))
df_bitacora_es.head()

## 5. Preparacion

- Cada fila es una **actividad** de una OT (una OT puede tener varias y abrirse en distintos dias).
- Se usa la **Hora de Inicio** de cada actividad: es el momento en que el tecnico empieza a trabajar y puede ir al almacen.
- Se quitan las actividades sin hora de inicio (~3%) y el dia de hoy (aun no termina).
- `matriz_dia_hora` arma una tabla con **una fila por dia** y **una columna por hora (0 a 23)**. Los dias u horas sin actividad cuentan como **0**: si no, los promedios saldrian inflados.

In [ ]:
FIN_ANALISIS = pd.Timestamp.today().normalize()

inicios = df_bitacora.dropna(subset=["activity_start_datetime"]).copy()
inicios = inicios[
    (inicios["activity_start_datetime"] >= INICIO_ANALISIS)
    & (inicios["activity_start_datetime"] < FIN_ANALISIS)
]
inicios["dia"] = inicios["activity_start_datetime"].dt.normalize()
inicios["hora"] = inicios["activity_start_datetime"].dt.hour

es_inspeccion = inicios["activity_name"].fillna("").str.upper().str.startswith("INSPECCI")
segmentos = {
    "Todas": inicios,
    "Sin inspecciones": inicios[~es_inspeccion],
    "Correctivas": inicios[inicios["maintenance_type"] == "corrective"],
}

dias = pd.date_range(inicios["dia"].min(), inicios["dia"].max(), freq="D")


def matriz_dia_hora(tabla):
    return (
        tabla.groupby(["dia", "hora"]).size()
        .unstack(fill_value=0)
        .reindex(index=dias, columns=range(24), fill_value=0)
    )


for nombre, tabla in segmentos.items():
    print(f"{nombre:18s}: {len(tabla):>7} actividades | {len(tabla) / len(dias):5.0f} por dia")
print(f"Dias analizados: {len(dias)} ({dias.min():%d/%m/%Y} al {dias.max():%d/%m/%Y})")

matriz = matriz_dia_hora(segmentos[SEGMENTO])     # la que usan las secciones 7 a 10

## 6. Perfil por hora

Para cada hora, cuantas actividades empiezan **por dia**:

- **Promedio:** un dia normal.
- **P90:** un dia cargado. En 9 de cada 10 dias hubo esa cantidad **o menos**. Para dimensionar personal se usa el P90: con el promedio, la mitad de los dias el almacen quedaria corto.

Se muestran los 3 grupos lado a lado para comparar.

In [ ]:
perfil = {}
for nombre, tabla in segmentos.items():
    m = matriz_dia_hora(tabla)
    perfil[(nombre, "Promedio")] = m.mean()
    perfil[(nombre, "P90")] = m.quantile(0.9)

df_perfil_hora = pd.DataFrame(perfil).round(1)
df_perfil_hora.index = [f"{h:02d}:00" for h in df_perfil_hora.index]
df_perfil_hora

## 7. Mapa de calor: dia de la semana x hora

Promedio de inicios por dia, separado por dia de la semana. **Blanco** = poca demanda, **rojo** = la hora mas cargada.

Sirve para ver que el patron cambia entre **lunes a viernes** y **fin de semana**. El color se calcula con una funcion propia, por eso no hace falta instalar `matplotlib`.

In [ ]:
df_mapa = matriz.groupby(matriz.index.day_of_week).mean().round(1)      # 0 = lunes ... 6 = domingo
df_mapa.index = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]
df_mapa.columns = [f"{h:02d}" for h in df_mapa.columns]

maximo = df_mapa.values.max()


def color(valor):
    t = valor / maximo                                  # 0 = sin actividad, 1 = la hora mas cargada
    verde, azul = int(255 - 155 * t), int(255 - 255 * t)
    texto = "white" if t > 0.6 else "black"
    return f"background-color: rgb(255,{verde},{azul}); color: {texto}"


print(f"{SEGMENTO}: inicios de actividad por hora (promedio por dia)")
df_mapa.style.map(color).format("{:.0f}")

## 8. Franjas de demanda

Se agrupan las horas en niveles, por separado para **lunes a viernes** y **sabado y domingo**. Se compara el P90 de cada hora con la hora mas cargada de ese tipo de dia:

| Nivel | Regla | Lectura para el almacen |
|---|---|---|
| **Alta** | 60% o mas de la hora pico | Reforzar personal |
| **Media** | entre 30% y 60% | Personal normal |
| **Baja** | menos de 30% | Personal minimo |

Las horas seguidas con el mismo nivel se juntan en una franja (por ejemplo 10:00 - 13:00).

In [ ]:
tipo_dia = pd.Series(matriz.index.day_of_week < 5, index=matriz.index).map(
    {True: "Lunes a viernes", False: "Sabado y domingo"}
)

franjas = []
niveles = {}                                            # nivel de cada hora, por tipo de dia (lo usa la seccion 9)
for tipo, matriz_tipo in matriz.groupby(tipo_dia):
    p90 = matriz_tipo.quantile(0.9)
    promedio = matriz_tipo.mean()
    nivel = pd.cut(p90 / p90.max(), bins=[-1, 0.3, 0.6, 2], labels=["Baja", "Media", "Alta"], right=False)
    niveles[tipo] = nivel

    bloque = (nivel != nivel.shift()).cumsum()          # cambia de numero cada vez que cambia el nivel
    for _, horas in nivel.groupby(bloque):
        h_ini, h_fin = horas.index.min(), horas.index.max()
        franjas.append({
            "Tipo de dia": tipo,
            "Franja": f"{h_ini:02d}:00 - {h_fin + 1:02d}:00",
            "Nivel": horas.iloc[0],
            "Inicios por hora (promedio)": round(promedio.loc[h_ini:h_fin].mean(), 1),   # .loc incluye h_fin
            "Inicios por hora (P90)": round(p90.loc[h_ini:h_fin].mean(), 1),
        })

df_franjas = pd.DataFrame(franjas)
df_franjas

## 9. Sustento: las horas punta son reales y se repiten

Para pedir mas personal no basta con un promedio: hay que mostrar que la punta **concentra** la demanda, que **se repite** casi todos los dias y que **no es de una temporada**. Todo se calcula para **lunes a viernes**, usando las horas de nivel **Alta**:

| Prueba | Pregunta que responde |
|---|---|
| **Concentracion** | ¿Que parte del trabajo del dia cae en esas pocas horas? |
| **Intensidad** | ¿Cuantas veces mas carga tiene la hora pico que una hora promedio? |
| **Dia cargado** | En un dia exigente (P90), ¿cada cuantos minutos empieza una actividad en la hora pico? |
| **Consistencia** | ¿En que % de los dias esas horas estuvieron realmente entre las mas cargadas? |
| **Estabilidad** | ¿El patron se mantiene mes a mes? |

In [ ]:
habiles = matriz[matriz.index.day_of_week < 5]
horas_alta = [h for h in range(24) if niveles["Lunes a viernes"][h] == "Alta"]
hora_pico = int(habiles.mean().idxmax())

# Concentracion: % de los inicios del dia que caen en horas Alta
pct_horas = len(horas_alta) / 24
pct_demanda = habiles[horas_alta].sum().sum() / habiles.sum().sum()

# Intensidad: hora pico frente al promedio por hora del dia
intensidad = habiles.mean().max() / habiles.mean().mean()

# Dia cargado: inicios en la hora pico en el P90 y cada cuantos minutos
p90_pico = habiles[hora_pico].quantile(0.9)
minutos_entre_inicios = 60 / p90_pico

# Consistencia: % de dias en que cada hora Alta estuvo entre las N horas mas cargadas de ese dia
ranking = habiles.rank(axis=1, ascending=False, method="min")
consistencia = (ranking[horas_alta] <= len(horas_alta)).mean()

# Estabilidad: inicios por dia en horas Alta, mes a mes.
# El primer mes (mayo 2025) fue el arranque del sistema: se muestra, pero no se usa para el minimo y el maximo.
df_estabilidad = (
    habiles[horas_alta].sum(axis=1)
    .groupby(habiles.index.to_period("M")).mean()
    .round(1).rename("Inicios por dia en horas Alta").to_frame()
)
estable = df_estabilidad.iloc[1:, 0]

df_sustento = pd.DataFrame({
    "Indicador": [
        "Horas de nivel Alta (lunes a viernes)",
        "% del dia que representan",
        "% de los inicios del dia que concentran",
        f"Hora pico",
        "Veces mas carga que una hora promedio",
        f"Inicios en la hora pico en un dia cargado (P90)",
        "Minutos entre inicios en la hora pico (dia cargado)",
        "Consistencia: % de dias en que fueron horas mas cargadas (promedio)",
        "Inicios por dia en horas Alta: mes mas bajo (sin el mes de arranque)",
        "Inicios por dia en horas Alta: mes mas alto (sin el mes de arranque)",
    ],
    "Valor": [
        ", ".join(f"{h:02d}:00" for h in horas_alta),
        f"{pct_horas:.0%}",
        f"{pct_demanda:.0%}",
        f"{hora_pico:02d}:00",
        f"{intensidad:.1f}",
        f"{p90_pico:.0f}",
        f"{minutos_entre_inicios:.1f}",
        f"{consistencia.mean():.0%}",
        f"{estable.min():.0f}",
        f"{estable.max():.0f}",
    ],
})

print("Consistencia por hora (% de dias habiles en que esa hora estuvo entre las mas cargadas):")
print((consistencia * 100).round(0).rename(lambda h: f"{h:02d}:00").to_string())
df_sustento

In [ ]:
df_estabilidad

## 10. Conclusiones

Texto armado con los resultados de las secciones anteriores. Se actualiza solo cada vez que se ejecuta el notebook.

**Ojo:** estos numeros miden **cuando empiezan las actividades** (la demanda). Para pasar a *cuantas personas* se necesitan faltan dos datos que esta bitacora no tiene: que % de actividades piden repuestos y cuantos minutos toma atender cada pedido. Se pueden sacar de las salidas del almacen (kardex).

In [ ]:
franjas_alta_lv = df_franjas[(df_franjas["Tipo de dia"] == "Lunes a viernes") & (df_franjas["Nivel"] == "Alta")]["Franja"]
franjas_alta_fs = df_franjas[(df_franjas["Tipo de dia"] == "Sabado y domingo") & (df_franjas["Nivel"] == "Alta")]["Franja"]

print(f"Analisis de {len(dias)} dias ({dias.min():%d/%m/%Y} al {dias.max():%d/%m/%Y}), grupo: {SEGMENTO}.")
print()
print(f"1. De lunes a viernes, las horas punta son {' y '.join(franjas_alta_lv)}.")
print(f"   Son el {pct_horas:.0%} del dia, pero concentran el {pct_demanda:.0%} de los inicios de actividad.")
print(f"2. La hora pico es las {hora_pico:02d}:00, con {intensidad:.1f} veces la carga de una hora promedio.")
print(f"   En un dia cargado empiezan {p90_pico:.0f} actividades en esa hora: una cada {minutos_entre_inicios:.1f} minutos.")
print(f"3. No es casualidad: esas horas estuvieron entre las mas cargadas en el {consistencia.mean():.0%} de los dias habiles.")
print(f"4. El patron se repite todos los meses: entre {estable.min():.0f} y {estable.max():.0f} "
      f"inicios por dia en horas punta (sin contar {df_estabilidad.index[0]}, mes de arranque del sistema).")
print(f"5. Sabado y domingo la punta cambia a {' y '.join(franjas_alta_fs)}.")

## 11. Exportar a Excel (opcional)

Cambiar `EXPORTAR_EXCEL` a `True` para guardar las tablas en `Horas_Punta_Almacen.xlsx` (una hoja por tabla), por ejemplo para presentarlas. Solo tiene datos agregados, sin nombres de tecnicos.

In [ ]:
EXPORTAR_EXCEL = False

if EXPORTAR_EXCEL:
    with pd.ExcelWriter("Horas_Punta_Almacen.xlsx") as writer:
        df_sustento.to_excel(writer, sheet_name="Sustento", index=False)
        df_franjas.to_excel(writer, sheet_name="Franjas", index=False)
        df_mapa.to_excel(writer, sheet_name="Mapa dia x hora")
        df_perfil_hora.to_excel(writer, sheet_name="Perfil por hora")
        df_estabilidad.to_excel(writer, sheet_name="Estabilidad mensual")
    print("Archivo guardado: Horas_Punta_Almacen.xlsx")